# Serve local Mistral on Colab GPU (T4)
Runtime -> Change runtime type -> **T4 GPU**, then Run All. Last cell prints the
URL for `MISTRAL_COLAB_URL=` in `~/.config/market-secrets/credentials.env`.

Uses Ollama (matches `ingest/mistral_extract.py`'s local backend exactly --
same API shape as `127.0.0.1:11434`, just tunneled) so no client-side code
change is needed beyond pointing at this URL instead of localhost. On a T4
this should run a `mistral` (7B) call in a few seconds instead of the
45-60s/page seen on Mac CPU.

In [ ]:
!nvidia-smi -L

In [ ]:
# Install and start Ollama, then pull the model onto the T4
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, os
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
server = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(5)
!ollama pull mistral

In [ ]:
# Expose via Cloudflare quick tunnel. NOTE: tunnel caps requests at ~100s --
# a T4 call should land well under that (unlike the ~50s/page CPU baseline
# this is meant to replace), but keep prompts to one page at a time.
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import subprocess, re, time
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 30:
    line = tun.stdout.readline()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
print('MISTRAL_COLAB_URL=' + (url or 'NOT FOUND -- re-run this cell'))

Copy the printed `MISTRAL_COLAB_URL=...` line into
`~/.config/market-secrets/credentials.env`, then run
`python3 -m ingest.mistral_extract --backend colab <file>`.

Keep the tab open while using the endpoint; free tier disconnects after a
few hours idle. Re-running the tunnel cell gives a new URL if it drops.